# Notebook to Compare Heuristics

In [16]:
import asyncio
import nest_asyncio

import pandas as pd
from time import time

import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

In [17]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)

In [18]:
nest_asyncio.apply()

In [19]:
from src.heuristics import (
    welfare_greedy,
    kempe_greedy,
    water_filling_greedy,
    marginal_packing,
    WelfareGRASP,
)

from src.diffusion_models import (
    estimate_cascade_influence,
    estimate_cascade_by_community,
)

In [20]:
from src import Loader
from pathlib import Path

path_to_networks = Path('../data/synthetic/networks/')
path_to_results = Path('../../results/barbasi_albert/size_1000/')

file_name = 'barbasi_albert_1000'

In [21]:
async def main():
    loader = Loader(max_workers=4)
    graph = await loader.load(f'{path_to_networks}/{file_name}.pkl')
    communities = list(greedy_modularity_communities(graph))
    costs = nx.get_node_attributes(graph, 'node_costs')

    return graph, communities, costs


graph, communities, costs = asyncio.run(main())

## Comparison of Welfare Greedy and Kempe Greedy

In [22]:
alpha_fixed = -2
p_fixed = 0.25
k_values = [1, 5, 10, 20]
num_sims = 1000

In [23]:
results_k_variation = []

for k in k_values:
    random.seed(42)
    np.random.seed(42)
    # Welfare-based approach
    start = time()
    welfare_seeds = welfare_greedy(
        graph=graph,
        communities=communities,
        k=k,
        alpha=alpha_fixed,
        probability=p_fixed,
        num_sims=num_sims,
    )
    welfare_time = time() - start

    welfare_influence = estimate_cascade_influence(
        graph=graph,
        seeds=welfare_seeds,
        probability=p_fixed,
        num_simulations=num_sims,
        random_state=42,
    )

    welfare_by_comm = estimate_cascade_by_community(
        graph=graph,
        seeds=welfare_seeds,
        probability=p_fixed,
        num_simulations=num_sims // 2,
        random_state=42,
    )
    welfare_by_comm_rounded = {comm: round(val, 2) for comm, val in welfare_by_comm.items()}

    # Kempe et al. original greedy approach
    start = time()
    kempe_seeds = kempe_greedy(
        graph=graph,
        k=k,
        probability=p_fixed,
        num_simulations=num_sims,
    )
    kempe_time = time() - start

    kempe_influence = estimate_cascade_influence(
        graph=graph,
        seeds=kempe_seeds,
        probability=p_fixed,
        num_simulations=num_sims,
        random_state=42,
    )

    kempe_by_comm = estimate_cascade_by_community(
        graph=graph,
        seeds=kempe_seeds,
        probability=p_fixed,
        num_simulations=num_sims // 2,
        random_state=42,
    )
    kempe_by_comm_rounded = {comm: round(val, 2) for comm, val in kempe_by_comm.items()}

    # Calculate utility gap and PoF
    utility_gap = max(welfare_by_comm.values()) - min(welfare_by_comm.values())

    welfare_ugap = max(welfare_by_comm.values()) - min(welfare_by_comm.values())
    kempe_ugap = max(kempe_by_comm.values()) - min(kempe_by_comm.values())

    pof = 1 - (welfare_influence / kempe_influence) if kempe_influence > 0 else 0

    results_k_variation.append(
        {
            'k': k,
            'alpha': alpha_fixed,
            'p': p_fixed,
            'kempe_seeds': kempe_seeds,
            'kempe_time_s': kempe_time,
            'kempe_total_influence': kempe_influence,
            'kempe_influence_by_community': kempe_by_comm_rounded,
            'kempe_utility_gap': kempe_ugap,
            'welfare_seeds': welfare_seeds,
            'welfare_time_s': welfare_time,
            'welfare_total_influence': welfare_influence,
            'welfare_influence_by_community': welfare_by_comm_rounded,
            'welfare_utility_gap': welfare_ugap,
            'price_of_fairness': pof,
        }
    )

Selecting seeds: 100%|██████████| 20/20 [03:22<00:00, 10.11s/it, seeds=20]


In [24]:
df_k_variation = pd.DataFrame(results_k_variation)
df_k_variation.to_csv(f'{path_to_results}/kempe_welfare_k_variation_{file_name}.csv', index=False)

In [25]:
k_fixed = 10  # fixed number of seeds
p_fixed = 0.25  # edge activation probability
alpha_values = [0.5, 0.0, -2.0, -5.0, -7.0, -9.0]  # varying inequality-aversion parameter
num_sims = 1000

In [26]:
results_alpha_variation = []

for alpha in alpha_values:
    random.seed(42)
    np.random.seed(42)
    start = time()
    kempe_seeds = kempe_greedy(
        graph=graph,
        k=k_fixed,
        probability=p_fixed,
        num_simulations=num_sims,
    )
    kempe_time = time() - start

    kempe_influence = estimate_cascade_influence(
        graph=graph,
        seeds=kempe_seeds,
        probability=p_fixed,
        num_simulations=num_sims,
        random_state=42,
    )

    kempe_by_comm = estimate_cascade_by_community(
        graph=graph,
        seeds=kempe_seeds,
        probability=p_fixed,
        num_simulations=num_sims // 2,
        random_state=42,
    )
    kempe_by_comm_rounded = {comm: round(val, 2) for comm, val in kempe_by_comm.items()}

    # Welfare-based approach with varying alpha
    start = time()
    welfare_seeds = welfare_greedy(
        graph=graph,
        communities=communities,
        k=k_fixed,
        alpha=alpha,
        probability=p_fixed,
        num_sims=num_sims,
    )
    welfare_time = time() - start

    welfare_influence = estimate_cascade_influence(
        graph=graph,
        seeds=welfare_seeds,
        probability=p_fixed,
        num_simulations=num_sims,
        random_state=42,
    )

    welfare_by_comm = estimate_cascade_by_community(
        graph=graph,
        seeds=welfare_seeds,
        probability=p_fixed,
        num_simulations=num_sims // 2,
        random_state=42,
    )
    welfare_by_comm_rounded = {comm: round(val, 2) for comm, val in welfare_by_comm.items()}

    # Calculate utility gap and PoF
    utility_gap = max(welfare_by_comm.values()) - min(welfare_by_comm.values())

    welfare_ugap = max(welfare_by_comm.values()) - min(welfare_by_comm.values())
    kempe_ugap = max(kempe_by_comm.values()) - min(kempe_by_comm.values())

    pof = 1 - (welfare_influence / kempe_influence) if kempe_influence > 0 else 0

    results_alpha_variation.append(
        {
            'k': k,
            'alpha': alpha,
            'p': p_fixed,
            'kempe_seeds': kempe_seeds,
            'kempe_time_s': kempe_time,
            'kempe_total_influence': kempe_influence,
            'kempe_influence_by_community': kempe_by_comm_rounded,
            'kempe_utility_gap': kempe_ugap,
            'welfare_seeds': welfare_seeds,
            'welfare_time_s': welfare_time,
            'welfare_total_influence': welfare_influence,
            'welfare_influence_by_community': welfare_by_comm_rounded,
            'welfare_utility_gap': welfare_ugap,
            'price_of_fairness': pof,
        }
    )

Selecting seeds: 100%|██████████| 10/10 [01:46<00:00, 10.64s/it, seeds=10, welfare=8462713025771.6172]                                                                              


In [27]:
df_alpha_variation = pd.DataFrame(results_alpha_variation)
df_alpha_variation.to_csv(f'{path_to_results}/kempe_welfare_alpha_variation_{file_name}.csv', index=False)

## Comparison of Different Models vs Welfare-based Greedy

In [28]:
k_fixed = 10  # number of seeds to select
p_fixed = 0.25  # edge activation probability
budget_fixed = 15.0  # Total budget available
alphas = [0.5, 0.1, 0.0, -2.0, -5.0, -7.0, -9.0]  # varying inequality-aversion parameter
num_sims = 1000

In [29]:
results_welfare_wfg = []

for alpha in alphas:
    random.seed(42)
    np.random.seed(42)

    start = time()
    welfare_seeds = welfare_greedy(
        graph=graph,
        communities=communities,
        k=k_fixed,
        alpha=alpha,
        probability=p_fixed,
        num_sims=num_sims,
    )
    welfare_time = time() - start

    welfare_influence = estimate_cascade_influence(
        graph=graph,
        seeds=welfare_seeds,
        probability=p_fixed,
        num_simulations=num_sims,
        random_state=42,
    )

    welfare_by_comm = estimate_cascade_by_community(
        graph=graph,
        seeds=welfare_seeds,
        probability=p_fixed,
        num_simulations=num_sims // 2,
        random_state=42,
    )
    welfare_by_comm_rounded = {comm: round(val, 2) for comm, val in welfare_by_comm.items()}
    welfare_ugap = max(welfare_by_comm.values()) - min(welfare_by_comm.values())

    random.seed(42)
    np.random.seed(42)

    start = time()
    wfg_seeds = water_filling_greedy(
        graph=graph,
        budget=budget_fixed,
        costs=costs,
        alpha=alpha,
        probability=p_fixed,
        num_sims=num_sims,
    )
    wfg_time = time() - start

    wfg_influence = estimate_cascade_influence(
        graph=graph,
        seeds=wfg_seeds,
        probability=p_fixed,
        num_simulations=num_sims,
        random_state=42,
    )

    wfg_by_comm = estimate_cascade_by_community(
        graph=graph,
        seeds=wfg_seeds,
        probability=p_fixed,
        num_simulations=num_sims // 2,
        random_state=42,
    )
    wfg_by_comm_rounded = {comm: round(val, 2) for comm, val in wfg_by_comm.items()}
    wfg_ugap = max(wfg_by_comm.values()) - min(wfg_by_comm.values())

    random.seed(42)
    np.random.seed(42)

    start = time()
    mpg_seeds = marginal_packing(
        graph=graph,
        costs=costs,
        budget=budget_fixed,
        alpha=alpha,
        probability=p_fixed,
        num_sims=num_sims,
    )
    mpg_time = time() - start

    mpg_influence = estimate_cascade_influence(
        graph=graph,
        seeds=mpg_seeds,
        probability=p_fixed,
        num_simulations=num_sims,
        random_state=42,
    )

    mpg_by_comm = estimate_cascade_by_community(
        graph=graph,
        seeds=mpg_seeds,
        probability=p_fixed,
        num_simulations=num_sims // 2,
        random_state=42,
    )
    mpg_by_comm_rounded = {comm: round(val, 2) for comm, val in mpg_by_comm.items()}
    mpg_ugap = max(mpg_by_comm.values()) - min(mpg_by_comm.values())

    start = time()
    grasp_solver = WelfareGRASP(
        graph=graph,
        costs=costs,
        budget=budget_fixed,
        welfare=alpha,
        propagation_rate=p_fixed,
        num_sims=num_sims,
        max_iter=50,
        max_evaluations=num_sims // 2,
        alpha=0.5,
    )
    grasp_seeds = grasp_solver.solve()
    grasp_time = time() - start

    grasp_influence = estimate_cascade_influence(
        graph=graph,
        seeds=grasp_seeds,
        probability=p_fixed,
        num_simulations=num_sims,
        random_state=42,
    )

    grasp_by_comm = estimate_cascade_by_community(
        graph=graph,
        seeds=grasp_seeds,
        probability=p_fixed,
        num_simulations=num_sims // 2,
        random_state=42,
    )
    grasp_by_comm_rounded = {comm: round(val, 2) for comm, val in grasp_by_comm.items()}
    grasp_ugap = max(grasp_by_comm.values()) - min(grasp_by_comm.values())
    grasp_seed_count = len(grasp_seeds)

    wfg_advantage = wfg_influence > welfare_influence
    mpg_advantage = mpg_influence > welfare_influence
    grasp_advantage = grasp_influence > welfare_influence

    welfare_seed_count = len(welfare_seeds) if isinstance(welfare_seeds, (list, set)) else k_fixed
    wfg_seed_count = len(wfg_seeds)
    mpg_seed_count = len(mpg_seeds)
    grasp_seed_count = len(grasp_seeds)

    results_welfare_wfg.append(
        {
            'alpha': alpha,
            'k': k_fixed,
            'p': p_fixed,
            'budget': budget_fixed,

            # Welfare-Greedy results
            'welfare_seeds': welfare_seeds,
            'welfare_seed_count': welfare_seed_count,
            'welfare_time_s': welfare_time,
            'welfare_total_influence': welfare_influence,
            'welfare_influence_by_community': welfare_by_comm_rounded,
            'utility_gap_welfare': welfare_ugap,

            # Water Filling Greedy results
            'wfg_seeds': wfg_seeds,
            'wfg_seed_count': wfg_seed_count,
            'wfg_time_s': wfg_time,
            'wfg_total_influence': wfg_influence,
            'wfg_influence_by_community': wfg_by_comm_rounded,
            'utility_gap_wfg': wfg_ugap,

            # Marginal Packing Greedy results
            'mpg_seeds': mpg_seeds,
            'mpg_seed_count': mpg_seed_count,
            'mpg_time_s': mpg_time,
            'mpg_total_influence': mpg_influence,
            'mpg_influence_by_community': mpg_by_comm_rounded,
            'utility_gap_mpg': mpg_ugap,

            'grasp_seeds': grasp_seeds,
            'grasp_seed_count': grasp_seed_count,
            'grasp_time_s': grasp_time,
            'grasp_total_influence': grasp_influence,
            'grasp_influence_by_community': grasp_by_comm_rounded,
            'utility_gap_grasp': grasp_ugap,

            'wfg_advantage': wfg_advantage,
            'mpg_advantage': mpg_advantage,
            'grasp_advantage': grasp_advantage,

            'total_execution_time_s': welfare_time + wfg_time + mpg_time + grasp_time,
            'time_ratio_grasp_vs_welfare': grasp_time / welfare_time if welfare_time > 0 else float('inf'),
            'time_ratio_wfg_vs_welfare': wfg_time / welfare_time if welfare_time > 0 else float('inf'),
            'time_ratio_mpg_vs_welfare': mpg_time / welfare_time if welfare_time > 0 else float('inf'),
        }
    )

Selecting seeds: 100%|██████████| 50/50 [00:12<00:00,  3.85it/s, seeds=4, cost=10.1]


In [30]:
df_welfare_wfg = pd.DataFrame(results_welfare_wfg)
output_filename = f'{path_to_results}/welfare_wfg_alpha_comparison_{file_name}.csv'
df_welfare_wfg.to_csv(output_filename, index=False)